# S5Mars: 20 confronti qualitativi per la Figura 5.1

Notebook **autosufficiente per Kaggle**, senza training e senza import dal repository.
Riusa le architetture, il resize, la normalizzazione e la palette di
`kaggle_segmentation_training.ipynb`; il download OAuth riprende il training e il notebook PTQ.

1. Estrae **20 immagini casuali distinte dal solo test S5Mars**, con seed 42, e salva la selezione.
2. Scarica da Google Drive il `best.ckpt` di un modello, predice le stesse 20 immagini in FP32,
   salva le mask, libera il modello e passa al successivo: **un solo modello sulla GPU**.
3. Salva 20 griglie **3 x 3**, nell'ordine della Figura 5.1: RGB, ground truth,
   U-Net/R34, U-Net/MV2, DeepLabV3/R34, DeepLabV3+/MV2, SegFormer-B0, LCNet3_7, LCNet3_11.
4. Mostra ciascun confronto separatamente; offre anche un navigatore per sfogliare e scegliere
   il preferito senza rifare inferenza. La scelta esporta i pannelli e la figura pronta per LaTeX.

**Avvio:** sessione Kaggle con Internet e GPU; abilita i Secrets
`GDRIVE_CLIENT_ID`, `GDRIVE_CLIENT_SECRET`, `GDRIVE_REFRESH_TOKEN`, come nel training.
Sono accettati anche gli alias `GOOGLE_CLIENT_ID`, `GOOGLE_CLIENT_SECRET`,
`GOOGLE_REFRESH_TOKEN` del PTQ. `HF_TOKEN` e' opzionale se il dataset e' accessibile senza login.
Esegui le celle in ordine. Gli ID delle sette cartelle sono gia' compilati,
verificati su Drive il 2026-10-06. Nessun upload o modifica su Drive.

Gli output vengono scritti in `/kaggle/working/s5mars_qualitative/seed_42_n20/`.
Per una nuova estrazione cambia `SEED`: avrai una directory distinta.


## Scelte e limiti dopo la revisione critica

- Stesse 20 immagini e preprocessing di valutazione per tutti i modelli: resize RGB bilineare,
  mask nearest-neighbour, 512 x 512, normalizzazione ImageNet, nessuna augmentation.
  I pannelli RGB mostrano esattamente l'immagine ridimensionata usata in inferenza.
- Batch 1 e FP32 (senza AMP/TF32) privilegiano coerenza con la baseline e memoria GPU contenuta;
  per 140 predizioni il costo e' accettabile. Si conservano solo ID uint8, non logits.
  Il dataset HF puo' scaricare/cacheare l'intero split test, ma vengono decodificate solo le 20 righe scelte.
- `best.ckpt` resta selezionato sulla validation. La scelta dell'immagine serve solo all'illustrazione:
  il campione di 20 immagini e la figura preferita **non sono nuove stime di accuratezza** e non
  modificano la scelta dei modelli. Non vengono cercati campioni con metriche migliori.
- Gli ID Drive delle LCNet puntano agli esperimenti della tesi, non al nuovo preset di training
  da 30 epoche. Epoca selezionata, budget di training (se presente), config e checksum sono salvati.
- Prima dell'inferenza controlliamo i sette file remoti; a ogni download controlliamo dimensione e MD5.
  Il caricamento dei pesi e' strict. Metadati incompatibili bloccano il modello; per i checkpoint
  storici con metadati parziali sono registrati i campi mancanti, con provenienza dalla cartella nota.
- Una riesecuzione riusa solo predizioni complete con le stesse immagini, gli stessi pesi, codice e
  versioni delle librerie. File incompleti/corrotti vengono rigenerati. Non cambiare configurazione
  dentro una directory gia' usata: per esperimenti diversi cambia `RUN_TAG` o `SEED`.
  I checkpoint locali restano in cache; lo ZIP finale li esclude per ridurre dimensione e duplicazioni.


## 1. Dipendenze
La stessa terna del training: **torch 2.9.1, torchvision 0.24.1, torchaudio 2.9.1**, tutte dall'indice CUDA 12.8. Transformers importa anche componenti audio durante il caricamento di SegFormer: il torchaudio preinstallato deve essere allineato a torch, anche se il notebook elabora soltanto immagini.

Dopo aver eseguito questa cella, **riavvia il kernel/sessione** per rimuovere le vecchie librerie native dalla memoria, poi esegui le celle dalla configurazione in avanti. Per correggere `undefined symbol: torch_library_impl`, riesegui questa installazione e riavvia prima della cella Runtime.

**SegFormer:** `transformers==4.49.0` mantiene la struttura dei layer del checkpoint (`encoder.patch_embeddings`, `encoder.block`, `encoder.layer_norm`). Le release 5.x recenti possono usare nomi diversi; installare una versione non fissata puo' impedire il caricamento strict dei pesi. Dopo il downgrade riavvia il kernel anche se la nuova versione risulta gia' installata.


In [ ]:
%pip install -q torch==2.9.1 torchvision==0.24.1 torchaudio==2.9.1 --index-url https://download.pytorch.org/whl/cu128
%pip install -q datasets huggingface_hub transformers==4.49.0 segmentation-models-pytorch pillow matplotlib tqdm google-api-python-client google-auth ipywidgets


## 2. Configurazione
Normalmente basta eseguire il notebook. `BATCH_SIZE = 1` tiene basso il consumo GPU; `DISPLAY_MODE = "interactive"` mostra una sola griglia alla volta.


In [ ]:
SEED = 42
NUM_SAMPLES = 20
BATCH_SIZE = 1
REPO_ID = "Mirali33/mb-s5mars"
DATASET_REVISION = "main"  # Il fingerprint effettivo e' salvato nella selezione.
TEST_SPLIT = "test"
IMAGE_SIZE = (512, 512)
IMAGE_MEAN = (0.485, 0.456, 0.406)
IMAGE_STD = (0.229, 0.224, 0.225)
NUM_CLASSES = 9
IGNORE_INDEX = -100
PRETRAINED_NAME = "nvidia/segformer-b0-finetuned-ade-512-512"
RUN_TAG = f"seed_{SEED}_n{NUM_SAMPLES}"
OUTPUT_BASE = "/kaggle/working/s5mars_qualitative"  # Locale: "outputs/s5mars_qualitative".
DEVICE = "cuda"  # Imposta "cpu" solo per debug; CPU molto piu' lenta.
RESUME_COMPLETED_MODELS = True
DISPLAY_MODE = "sequential"  # [sequential, interactive]: 20 output separati o navigatore.
FIGURE_DPI = 180
CREATE_ZIP = True  # Include figure, pannelli, mask e manifest; esclude checkpoint.

CLASS_NAMES = {
    0: "Background", 1: "Bedrock", 2: "Hole", 3: "Ridge", 4: "Rock",
    5: "Rover", 6: "Sand / Soil", 7: "Sky", 8: "Track",
}
PALETTE = {
    0: (30, 30, 30), 1: (166, 118, 29), 2: (0, 109, 119),
    3: (131, 56, 236), 4: (230, 57, 70), 5: (255, 183, 3),
    6: (42, 157, 143), 7: (69, 123, 157), 8: (244, 162, 97),
}
# L'ordine segue esattamente i sette pannelli modello della Figura 5.1.
MODELS = [
    dict(key="smp_unet_resnet34", title="U-Net / ResNet34", family="smp",
         architecture="unet", encoder="resnet34",
         folder_id="1IJQfHvlQs_g8-uhVWHqWQD0WjkjP6jXg"),
    dict(key="smp_unet_mobilenet_v2", title="U-Net / MobileNetV2", family="smp",
         architecture="unet", encoder="mobilenet_v2",
         folder_id="1Ub1bDNDwNMTm5NSLnyIC6DVm5IRcLCmE"),
    dict(key="smp_deeplabv3_resnet34", title="DeepLabV3 / ResNet34", family="smp",
         architecture="deeplabv3", encoder="resnet34",
         folder_id="113m70R3G5WLJMTLwuUHAOlHT5eLe7lpX"),
    dict(key="smp_deeplabv3plus_mobilenet_v2", title="DeepLabV3+ / MobileNetV2", family="smp",
         architecture="deeplabv3plus", encoder="mobilenet_v2",
         folder_id="15gfIfMeV9Y_KjGjNzA7wizgT4W1GV60o"),
    dict(key="segformer_b0", title="SegFormer-B0", family="segformer_b0",
         folder_id="1gor4_Lct_xBMJuWlfxnAXm0fYNjcKBRs"),
    dict(key="lcnet3_7", title="LCNet$_{3,7}$", family="lcnet", variant="lcnet3_7",
         folder_id="1mdGw6tRVjZvb8BNxuAg0xjGJuhU1dw-6"),
    dict(key="lcnet3_11", title="LCNet$_{3,11}$", family="lcnet", variant="lcnet3_11",
         folder_id="198fJ76qxSjY6PEZ8guMjscVMDuDzGSE5"),
]


## 3. Runtime e salvataggi
I file di output sono locali alla sessione Kaggle e vanno scaricati prima di chiuderla.


In [ ]:
import gc
import hashlib
import importlib.metadata
import io
import json
import os
import random
import shutil
import warnings
import zipfile
from collections.abc import Sequence
from pathlib import Path

EXPECTED_TORCH_STACK = {"torch": "2.9.1", "torchvision": "0.24.1", "torchaudio": "2.9.1"}
EXPECTED_TRANSFORMERS_VERSION = "4.49.0"


def validate_torch_stack_versions():
    # Check installed metadata before importing native extensions or Transformers.
    mismatches = {}
    for package, expected in {**EXPECTED_TORCH_STACK, "transformers": EXPECTED_TRANSFORMERS_VERSION}.items():
        try:
            actual = importlib.metadata.version(package)
        except importlib.metadata.PackageNotFoundError:
            actual = "non installato"
        if actual.split("+")[0] != expected:
            mismatches[package] = {"expected": expected, "installed": actual}
    if mismatches:
        raise RuntimeError(
            f"Dipendenze non allineate: {mismatches}. "
            "Riesegui la cella Dipendenze (torch/torchvision/torchaudio/transformers), "
            "poi riavvia il kernel/sessione prima di proseguire."
        )


validate_torch_stack_versions()

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
from PIL import Image
import torch
if torch.__version__.split("+")[0] != EXPECTED_TORCH_STACK["torch"]:
    raise RuntimeError("torch in memoria non coincide con la versione installata: riavvia il kernel/sessione.")
import torchvision
import torchaudio
for package, module in (("torchvision", torchvision), ("torchaudio", torchaudio)):
    if module.__version__.split("+")[0] != EXPECTED_TORCH_STACK[package]:
        raise RuntimeError(f"{package} in memoria ha una versione precedente: riavvia il kernel/sessione.")
import torch.nn as nn
import torch.nn.functional as F
import segmentation_models_pytorch as smp
from datasets import load_dataset
import transformers
if transformers.__version__ != EXPECTED_TRANSFORMERS_VERSION:
    raise RuntimeError(
        f"Transformers in memoria: {transformers.__version__}; richiesto {EXPECTED_TRANSFORMERS_VERSION}. "
        "Riavvia il kernel/sessione dopo l'installazione prima di caricare SegFormer."
    )
from transformers import SegformerConfig, SegformerForSemanticSegmentation
from tqdm.auto import tqdm
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build as build_google_api
from googleapiclient.http import MediaIoBaseDownload
from IPython.display import display, FileLink, Image as DisplayImage, clear_output

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("Abilita una GPU nelle impostazioni Kaggle, poi riavvia la sessione.")
if NUM_SAMPLES <= 0 or BATCH_SIZE <= 0:
    raise ValueError("NUM_SAMPLES e BATCH_SIZE devono essere positivi")
if NUM_CLASSES != 9 or IMAGE_MEAN != (0.485, 0.456, 0.406) or IMAGE_STD != (0.229, 0.224, 0.225):
    raise ValueError("Mantieni classi e normalizzazione del training S5Mars")
if DISPLAY_MODE not in {"sequential", "interactive"}:
    raise ValueError("DISPLAY_MODE deve essere sequential o interactive")
if Path(RUN_TAG).name != RUN_TAG or RUN_TAG in {".", ".."}:
    raise ValueError("RUN_TAG deve essere un semplice nome di directory")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
device = torch.device(DEVICE)
OUTPUT_DIR = Path(OUTPUT_BASE) / RUN_TAG
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_VERSIONS = {
    name: importlib.metadata.version(name)
    for name in ("torch", "torchvision", "torchaudio", "transformers", "segmentation-models-pytorch", "numpy", "Pillow", "datasets")
}

def file_digest(path, algorithm="sha256"):
    digest = hashlib.new(algorithm)
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def canonical_digest(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".part")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    temporary.replace(path)

def save_png(path, array):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".part")
    Image.fromarray(np.asarray(array, dtype=np.uint8)).save(temporary, format="PNG")
    temporary.replace(path)

print("Device:", device, "| Output:", OUTPUT_DIR)
print("Versioni:", RUNTIME_VERSIONS)


## 4. Google Drive: controllo dei sette checkpoint
OAuth e Secrets come nei notebook precedenti. Qui leggiamo soltanto i metadati; i pesi saranno scaricati uno alla volta durante l'inferenza.


In [ ]:
def get_secret(name, aliases=(), required=True):
    for candidate in (name, *aliases):
        value = os.environ.get(candidate)
        if not value:
            try:
                from kaggle_secrets import UserSecretsClient
                value = UserSecretsClient().get_secret(candidate)
            except Exception:
                value = None
        if value:
            return value
    if required:
        raise RuntimeError(f"Secret mancante: {name}; abilitalo nelle impostazioni Kaggle.")
    return None

def build_drive_service():
    credentials = Credentials(
        token=None,
        refresh_token=get_secret("GDRIVE_REFRESH_TOKEN", ("GOOGLE_REFRESH_TOKEN",)),
        token_uri="https://oauth2.googleapis.com/token",
        client_id=get_secret("GDRIVE_CLIENT_ID", ("GOOGLE_CLIENT_ID",)),
        client_secret=get_secret("GDRIVE_CLIENT_SECRET", ("GOOGLE_CLIENT_SECRET",)),
    )
    return build_google_api("drive", "v3", credentials=credentials, cache_discovery=False)

def resolve_checkpoint(service, spec):
    response = service.files().list(
        q=(f"'{spec['folder_id']}' in parents and name = 'best.ckpt' "
           "and mimeType != 'application/vnd.google-apps.folder' and trashed = false"),
        fields="nextPageToken,files(id,name,size,md5Checksum,modifiedTime)",
        pageSize=100, supportsAllDrives=True, includeItemsFromAllDrives=True,
    ).execute()
    matches = response.get("files", [])
    if len(matches) != 1 or response.get("nextPageToken"):
        raise RuntimeError(f"{spec['key']}: atteso un solo best.ckpt, trovati {len(matches)}")
    remote = matches[0]
    if not remote.get("size") or not remote.get("md5Checksum"):
        raise RuntimeError(f"{spec['key']}: dimensione o checksum Drive mancanti")
    return remote

def download_checkpoint(service, spec, remote):
    destination = OUTPUT_DIR / "checkpoints" / spec["key"] / "best.ckpt"
    destination.parent.mkdir(parents=True, exist_ok=True)
    if (destination.is_file() and destination.stat().st_size == int(remote["size"])
            and file_digest(destination, "md5") == remote["md5Checksum"]):
        print("Checkpoint in cache:", spec["key"])
        return destination
    temporary = destination.with_suffix(".ckpt.part")
    try:
        request = service.files().get_media(fileId=remote["id"], supportsAllDrives=True)
        with temporary.open("wb") as output_file:
            downloader = MediaIoBaseDownload(output_file, request, chunksize=8 * 1024 * 1024)
            done = False
            while not done:
                _, done = downloader.next_chunk(num_retries=3)
        if temporary.stat().st_size != int(remote["size"]):
            raise IOError(f"{spec['key']}: download incompleto")
        if file_digest(temporary, "md5") != remote["md5Checksum"]:
            raise IOError(f"{spec['key']}: checksum del download non valido")
        temporary.replace(destination)
    finally:
        temporary.unlink(missing_ok=True)
    return destination

drive_service = build_drive_service()
REMOTE_CHECKPOINTS = {spec["key"]: resolve_checkpoint(drive_service, spec) for spec in MODELS}
for spec in MODELS:
    remote = REMOTE_CHECKPOINTS[spec["key"]]
    print(f"{spec['key']}: best.ckpt, {int(remote['size']) / 1024**2:.1f} MiB")


## 5. Preprocessing del training
Codice riusato dal notebook di training. Il transform viene costruito con tutte le probabilita' di augmentation a zero.


In [ ]:
class SegmentationTransform:
    def __init__(
        self,
        size,
        horizontal_flip_prob=0.0,
        vertical_flip_prob=0.0,
        random_rotate90_prob=0.0,
    ):
        self.size = size
        self.horizontal_flip_prob = float(horizontal_flip_prob)
        self.vertical_flip_prob = float(vertical_flip_prob)
        self.random_rotate90_prob = float(random_rotate90_prob)
        probabilities = {
            "horizontal_flip_prob": self.horizontal_flip_prob,
            "vertical_flip_prob": self.vertical_flip_prob,
            "random_rotate90_prob": self.random_rotate90_prob,
        }
        invalid = {name: value for name, value in probabilities.items() if not 0.0 <= value <= 1.0}
        if invalid:
            raise ValueError(f"Augmentation probabilities must be in [0, 1], got {invalid}")

    def __call__(self, image, mask):
        image = image.resize((self.size[1], self.size[0]), Image.Resampling.BILINEAR)
        mask = mask.resize((self.size[1], self.size[0]), Image.Resampling.NEAREST)

        # Keep worker outputs compact and non-Torch: RGB uint8 [H, W, 3] and
        # class-ID uint8 [H, W]. Conversion and normalization happen in main.
        image_array = np.asarray(image, dtype=np.uint8).copy()
        mask_array = np.asarray(mask, dtype=np.uint8).copy()
        if mask_array.ndim == 3:
            mask_array = mask_array[..., 0]

        # Draw each decision once and apply the same geometry to image and mask.
        if random.random() < self.horizontal_flip_prob:
            image_array = np.flip(image_array, axis=1)
            mask_array = np.flip(mask_array, axis=1)
        if random.random() < self.vertical_flip_prob:
            image_array = np.flip(image_array, axis=0)
            mask_array = np.flip(mask_array, axis=0)
        if random.random() < self.random_rotate90_prob:
            # Match Albumentations RandomRotate90: a sampled factor of 0 is an identity.
            quarter_turns = random.randrange(4)
            if quarter_turns:
                image_array = np.rot90(image_array, k=quarter_turns, axes=(0, 1))
                mask_array = np.rot90(mask_array, k=quarter_turns, axes=(0, 1))

        # flip/rot90 may return negative-stride views, which cannot be passed to Torch.
        return np.ascontiguousarray(image_array), np.ascontiguousarray(mask_array)

def prepare_batch(batch, device):
    images = torch.from_numpy(batch["image"]).permute(0, 3, 1, 2).contiguous()
    masks = torch.from_numpy(batch["mask"])
    # NumPy payloads are copied to CUDA without PyTorch multiprocessing storage.
    images = images.to(device).float().div_(255.0)
    masks = masks.to(device).long()
    mean = images.new_tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = images.new_tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    images.sub_(mean).div_(std)
    return images, masks

def colorize_mask(mask):
    mask = mask.cpu().numpy() if isinstance(mask, torch.Tensor) else np.asarray(mask)
    color = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for class_id, rgb in PALETTE.items():
        color[mask == class_id] = rgb
    return color


## 6. Scegliere e salvare le 20 immagini
La selezione e' senza reinserimento e precede ogni caricamento di modello. Il manifest conserva gli indici originali del test; non si usano train o validation.


In [ ]:
def prepare_selection(dataset):
    if not {"image", "mask"}.issubset(dataset.column_names):
        raise ValueError("Il test S5Mars deve contenere image e mask")
    if len(dataset) < NUM_SAMPLES:
        raise ValueError(f"Il test ha solo {len(dataset)} immagini; richieste {NUM_SAMPLES}")
    indices = np.random.default_rng(SEED).choice(len(dataset), size=NUM_SAMPLES, replace=False).tolist()
    settings = dict(
        schema_version=1, repo_id=REPO_ID, revision=DATASET_REVISION, split=TEST_SPLIT,
        dataset_fingerprint=dataset._fingerprint, test_size=len(dataset), seed=SEED,
        num_samples=NUM_SAMPLES, image_size=list(IMAGE_SIZE), mean=list(IMAGE_MEAN),
        std=list(IMAGE_STD), indices=indices,
        class_names={str(k): v for k, v in CLASS_NAMES.items()},
        palette={str(k): list(v) for k, v in PALETTE.items()},
    )
    manifest_path = OUTPUT_DIR / "selection.json"
    previous = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else None
    if previous is not None and previous["settings"] != settings:
        raise RuntimeError("Selezione/configurazione diversa: scegli un nuovo RUN_TAG o OUTPUT_BASE.")
    transform = SegmentationTransform(IMAGE_SIZE)
    items = []
    for order, index in enumerate(tqdm(indices, desc="Preparazione delle immagini"), start=1):
        row = dataset[int(index)]
        if not isinstance(row["image"], Image.Image) or not isinstance(row["mask"], Image.Image):
            raise TypeError("Le colonne image e mask devono contenere immagini PIL come nel training")
        image = row["image"].convert("RGB")
        mask = row["mask"].convert("L")
        if image.size != mask.size:
            raise ValueError(f"Test index {index}: dimensioni RGB e mask diverse")
        invalid_ids = np.setdiff1d(np.unique(np.asarray(mask)), np.arange(NUM_CLASSES))
        if invalid_ids.size:
            raise ValueError(f"Test index {index}: ID mask non validi {invalid_ids.tolist()}")
        rgb, labels = transform(image, mask)
        sample_id = f"sample_{order:02d}_test_{index:05d}"
        folder = OUTPUT_DIR / "samples" / sample_id
        paths = dict(rgb=folder / "rgb.png", ground_truth_ids=folder / "ground_truth_ids.png",
                     ground_truth_color=folder / "ground_truth_color.png")
        for name, array in (("rgb", rgb), ("ground_truth_ids", labels),
                            ("ground_truth_color", colorize_mask(labels))):
            save_png(paths[name], array)
        items.append(dict(
            order=order, test_index=int(index), sample_id=sample_id,
            original_size=list(image.size),
            paths={name: str(path.relative_to(OUTPUT_DIR).as_posix()) for name, path in paths.items()},
            sha256={name: file_digest(path) for name, path in paths.items()},
        ))
    manifest = dict(settings=settings, samples=items)
    if previous is not None and previous != manifest:
        raise RuntimeError("Contenuto delle immagini cambiato: scegli un nuovo RUN_TAG.")
    save_json(manifest_path, manifest)
    return manifest

HF_TOKEN = get_secret("HF_TOKEN", required=False)
test_dataset = load_dataset(REPO_ID, split=TEST_SPLIT, revision=DATASET_REVISION, token=HF_TOKEN)
selection = prepare_selection(test_dataset)
samples = selection["samples"]
SELECTION_SHA256 = canonical_digest(selection)
del test_dataset
gc.collect()
print("Indici del test selezionati:", selection["settings"]["indices"])
print("Selezione salvata:", OUTPUT_DIR / "selection.json")


## 7. Architetture riusate dal training
LCNet e wrapper SMP sono copiati dal training. SegFormer usa la stessa architettura B0 e interpolazione dei logits; inizializza solo la configurazione, senza scaricare inutilmente i pesi ADE20K/ImageNet che il checkpoint sostituisce integralmente.


In [ ]:
VARIANT_DILATIONS: dict[str, tuple[tuple[int, ...], tuple[int, ...]]] = {
    "lcnet3_7": ((2, 2, 2), (4, 4, 8, 8, 16, 16, 32)),
    "lcnet3_11": ((2, 2, 2), (4, 4, 8, 8, 16, 16, 32, 32, 32, 32, 32)),
}

class ConvBNSelu(nn.Module):
    """Convolution optionally followed by BatchNorm(eps=1e-3) and SELU."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int | tuple[int, int],
        stride: int | tuple[int, int] = 1,
        padding: int | tuple[int, int] = 0,
        dilation: int | tuple[int, int] = 1,
        groups: int = 1,
        norm_activation: bool = False,
    ) -> None:
        super().__init__()
        layers: list[nn.Module] = [
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                dilation=dilation,
                groups=groups,
                bias=False,
            )
        ]
        if norm_activation:
            layers.extend(
                [nn.BatchNorm2d(out_channels, eps=1e-3), nn.SELU(inplace=True)]
            )
        self.layers = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.layers(x)

class DownsamplingBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        if out_channels <= in_channels:
            raise ValueError("LCNet downsampling requires out_channels > in_channels")
        self.convolution = ConvBNSelu(
            in_channels, out_channels - in_channels, 3, stride=2, padding=1
        )
        self.pool = nn.MaxPool2d(2, stride=2)
        self.norm_activation = nn.Sequential(
            nn.BatchNorm2d(out_channels, eps=1e-3), nn.SELU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.norm_activation(torch.cat((self.convolution(x), self.pool(x)), dim=1))

class ThreeBranchContextAggregation(nn.Module):
    def __init__(self, channels: int, dilation: int) -> None:
        super().__init__()
        self.base = ConvBNSelu(channels, channels, 3, padding=1, norm_activation=True)
        self.local_depthwise = ConvBNSelu(
            channels, channels, 3, padding=1, groups=channels, norm_activation=True
        )
        self.dilated_depthwise = ConvBNSelu(
            channels,
            channels,
            3,
            padding=dilation,
            dilation=dilation,
            groups=channels,
            norm_activation=True,
        )
        self.norm_activation = nn.Sequential(
            nn.BatchNorm2d(channels, eps=1e-3), nn.SELU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        base = self.base(x)
        combined = base + self.local_depthwise(base) + self.dilated_depthwise(base)
        return self.norm_activation(combined)

class PartialChannelTransformation(nn.Module):
    def __init__(self, channels: int, dilation: int, partial_rate: float = 0.5) -> None:
        super().__init__()
        if not 0.0 < partial_rate < 1.0:
            raise ValueError(f"partial_rate must be in (0, 1), got {partial_rate}")
        self.identity_channels = round(channels * (1.0 - partial_rate))
        transformed_channels = channels - self.identity_channels
        self.tca = ThreeBranchContextAggregation(transformed_channels, dilation)
        self.pointwise = ConvBNSelu(
            channels, channels, 1, norm_activation=True
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity, transformed = torch.split(
            x, (self.identity_channels, x.shape[1] - self.identity_channels), dim=1
        )
        transformed = self.tca(transformed)
        return self.pointwise(torch.cat((identity, transformed), dim=1))

class DualAttentionGuidedDecoder(nn.Module):
    def __init__(self, deep_channels: int, shallow_channels: int, num_classes: int) -> None:
        super().__init__()
        self.deep_projection = ConvBNSelu(
            deep_channels, shallow_channels, 1, norm_activation=True
        )
        self.negative_projection = ConvBNSelu(
            shallow_channels, shallow_channels, 1, norm_activation=True
        )
        self.depthwise = ConvBNSelu(
            shallow_channels,
            shallow_channels,
            3,
            padding=1,
            groups=shallow_channels,
            norm_activation=True,
        )
        self.classifier = ConvBNSelu(
            shallow_channels, num_classes, 1, norm_activation=True
        )

    def forward(self, shallow: torch.Tensor, deep: torch.Tensor) -> torch.Tensor:
        spatial_map = torch.sigmoid(shallow)
        projected = self.deep_projection(deep)
        projected_map = torch.sigmoid(projected)
        negative_map = self.negative_projection(-projected_map)
        reverse_guided = negative_map * projected_map + projected
        reverse_guided = F.interpolate(
            reverse_guided,
            size=spatial_map.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        fused = self.depthwise(spatial_map * reverse_guided)
        return self.classifier(fused)

class LCNet(nn.Module):
    """LCNet3_7 or LCNet3_11 returning full-resolution raw logits."""

    def __init__(
        self,
        num_classes: int,
        variant: str = "lcnet3_7",
        in_channels: int = 3,
        base_channels: int = 32,
        partial_rate: float = 0.5,
        stage1_blocks: int | None = None,
        stage2_blocks: int | None = None,
        pretrained: bool = False,
        log_shapes: bool = False,
    ) -> None:
        super().__init__()
        variant = variant.lower()
        if variant not in VARIANT_DILATIONS:
            choices = ", ".join(VARIANT_DILATIONS)
            raise ValueError(f"Unsupported LCNet variant '{variant}'. Expected one of: {choices}.")
        if in_channels != 3:
            raise ValueError(f"LCNet requires RGB input (in_channels=3), got {in_channels}")
        if pretrained:
            raise ValueError("Pretrained weights are not available for this LCNet reimplementation")

        stage1_dilations, stage2_dilations = VARIANT_DILATIONS[variant]
        expected_blocks = (len(stage1_dilations), len(stage2_dilations))
        configured_blocks = (
            expected_blocks[0] if stage1_blocks is None else stage1_blocks,
            expected_blocks[1] if stage2_blocks is None else stage2_blocks,
        )
        if configured_blocks != expected_blocks:
            raise ValueError(
                f"Variant {variant} requires stage blocks {expected_blocks}, got {configured_blocks}"
            )

        self.variant = variant
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.log_shapes = log_shapes
        self.initialization = nn.Sequential(
            ConvBNSelu(in_channels, base_channels, 3, stride=2, padding=1, norm_activation=True),
            ConvBNSelu(base_channels, base_channels, 3, padding=1, norm_activation=True),
            ConvBNSelu(base_channels, base_channels, 3, padding=1, norm_activation=True),
        )
        self.stage1 = self._make_stage(
            base_channels, base_channels * 2, stage1_dilations, partial_rate
        )
        self.stage2 = self._make_stage(
            base_channels * 2, base_channels * 4, stage2_dilations, partial_rate
        )
        self.decoder = DualAttentionGuidedDecoder(
            base_channels * 4, base_channels * 2, num_classes
        )

    @staticmethod
    def _make_stage(
        in_channels: int,
        out_channels: int,
        dilations: tuple[int, ...],
        partial_rate: float,
    ) -> nn.Sequential:
        return nn.Sequential(
            DownsamplingBlock(in_channels, out_channels),
            *(PartialChannelTransformation(out_channels, d, partial_rate) for d in dilations),
        )

    def forward(self, images: torch.Tensor) -> torch.Tensor:
        if images.ndim != 4 or images.shape[1] != self.in_channels:
            raise ValueError(
                f"Expected images [B, {self.in_channels}, H, W], got {tuple(images.shape)}"
            )
        input_size = images.shape[-2:]
        shallow = self.stage1(self.initialization(images))
        deep = self.stage2(shallow)
        scores = self.decoder(shallow, deep)
        logits = F.interpolate(scores, size=input_size, mode="bilinear", align_corners=False)
        if self.log_shapes:
            print(f"logits: shape={tuple(logits.shape)}, dtype={logits.dtype}")
        return logits

    def apply_freeze(self, freeze: str) -> None:
        freeze = freeze.lower()
        for parameter in self.parameters():
            parameter.requires_grad = True
        if freeze == "none":
            return
        if freeze == "encoder":
            for module in (self.initialization, self.stage1, self.stage2):
                for parameter in module.parameters():
                    parameter.requires_grad = False
            return
        if freeze == "classifier":
            for parameter in self.parameters():
                parameter.requires_grad = False
            for parameter in self.decoder.classifier.parameters():
                parameter.requires_grad = True
            return
        raise ValueError(f"Unknown freeze mode '{freeze}'. Expected one of: none, encoder, classifier.")

class SMPModelForMars(nn.Module):
    def __init__(self, architecture, encoder_name, encoder_weights="imagenet", in_channels=3, num_classes=9, log_shapes=False):
        super().__init__()
        self.architecture = architecture.lower()
        self.encoder_name = encoder_name
        self.encoder_weights = encoder_weights
        self.in_channels = in_channels
        self.num_classes = num_classes
        self.log_shapes = log_shapes
        self.model = self._build_model()

    def _build_model(self):
        kwargs = dict(
            encoder_name=self.encoder_name,
            encoder_weights=self.encoder_weights,
            in_channels=self.in_channels,
            classes=self.num_classes,
            activation=None,
        )
        if self.architecture == "unet":
            return smp.Unet(**kwargs)
        if self.architecture == "deeplabv3":
            return smp.DeepLabV3(**kwargs)
        if self.architecture == "deeplabv3plus":
            return smp.DeepLabV3Plus(**kwargs)
        raise ValueError("SMP_ARCHITECTURE must be one of: unet, deeplabv3, deeplabv3plus")

    def forward(self, images):
        if images.ndim != 4:
            raise ValueError(f"Expected images [B, 3, H, W], got {images.shape}")
        if images.shape[1] != self.in_channels:
            raise ValueError(f"Expected {self.in_channels} input channels, got {images.shape[1]}")
        logits = self.model(images)
        if logits.shape[-2:] != images.shape[-2:]:
            raise ValueError(f"Expected logits spatial shape {images.shape[-2:]}, got {logits.shape[-2:]}")
        if self.log_shapes:
            print(f"logits: shape={tuple(logits.shape)}, dtype={logits.dtype}")
        return logits

    def apply_freeze(self, freeze):
        freeze = freeze.lower()
        for param in self.parameters():
            param.requires_grad = True
        if freeze == "none":
            return
        if freeze == "encoder":
            for param in self.model.encoder.parameters():
                param.requires_grad = False
            if hasattr(self.model, "decoder"):
                for param in self.model.decoder.parameters():
                    param.requires_grad = True
            for param in self.model.segmentation_head.parameters():
                param.requires_grad = True
            return
        if freeze == "classifier":
            for param in self.parameters():
                param.requires_grad = False
            for param in self.model.segmentation_head.parameters():
                param.requires_grad = True
            return
        raise ValueError(f"Unknown freeze mode: {freeze}. Expected one of: none, encoder, classifier.")

class SegFormerB0ForMars(nn.Module):
    def __init__(self, num_classes, ignore_index):
        super().__init__()
        config = SegformerConfig(
            num_labels=num_classes, semantic_loss_ignore_index=ignore_index,
            hidden_sizes=[32, 64, 160, 256], depths=[2, 2, 2, 2],
            num_attention_heads=[1, 2, 5, 8], sr_ratios=[8, 4, 2, 1],
            decoder_hidden_size=256,
        )
        self.model = SegformerForSemanticSegmentation(config)

    def forward(self, images):
        raw_logits = self.model(pixel_values=images).logits
        return F.interpolate(raw_logits, size=images.shape[-2:], mode="bilinear", align_corners=False)


def build_model(spec):
    if spec["family"] == "smp":
        return SMPModelForMars(spec["architecture"], spec["encoder"], encoder_weights=None,
                               in_channels=3, num_classes=NUM_CLASSES, log_shapes=False)
    if spec["family"] == "segformer_b0":
        return SegFormerB0ForMars(NUM_CLASSES, IGNORE_INDEX)
    if spec["family"] == "lcnet":
        return LCNet(num_classes=NUM_CLASSES, variant=spec["variant"],
                     in_channels=3, base_channels=32, partial_rate=0.5,
                     pretrained=False, log_shapes=False)
    raise ValueError(f"Famiglia non supportata: {spec['family']}")


## 8. Caricamento strict e verifica dei metadati
Non viene ripristinato alcun optimizer. I metadati disponibili devono coincidere con S5Mars e con l'architettura richiesta; i campi storici assenti vengono elencati nel manifest.


In [ ]:
def validate_checkpoint_config(config, spec):
    if not isinstance(config, dict):
        raise TypeError("config del checkpoint deve essere un dizionario")
    expected = dict(model_name=spec["family"], model_run_name=spec["key"],
                    num_classes=NUM_CLASSES, dataset_name="s5mars", repo_id=REPO_ID,
                    image_size=list(IMAGE_SIZE), sampling_tag="standard_sampling",
                    criterion_weight_type="square", augmentation_tag="no_aug")
    if spec["family"] == "smp":
        expected.update(smp_architecture=spec["architecture"], smp_encoder_name=spec["encoder"])
    if spec["family"] == "lcnet":
        expected.update(lcnet_variant=spec["variant"], lcnet_base_channels=32, lcnet_partial_rate=0.5)
    if spec["family"] == "segformer_b0":
        expected.update(pretrained_name=PRETRAINED_NAME)
    missing, mismatches = [], {}
    for key, value in expected.items():
        actual = config.get(key)
        if actual is None:
            missing.append(key)
        else:
            actual = list(actual) if key == "image_size" else actual
            if actual != value:
                mismatches[key] = dict(expected=value, checkpoint=actual)
    if config.get("class_names") is not None:
        actual_names = {int(k): v for k, v in config["class_names"].items()}
        if actual_names != CLASS_NAMES:
            mismatches["class_names"] = dict(expected=CLASS_NAMES, checkpoint=actual_names)
    if mismatches:
        raise ValueError(f"{spec['key']}: checkpoint incompatibile: {mismatches}")
    if missing:
        warnings.warn(f"{spec['key']}: metadati storici mancanti {missing}; "
                      "provenienza dalla cartella Drive configurata e pesi verificati strict.")
    return missing

def load_trained_model(path, spec):
    # Checkpoint dei propri esperimenti, come nei notebook training/PTQ.
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(checkpoint, dict) or "model_state_dict" not in checkpoint:
        raise ValueError(f"{spec['key']}: manca model_state_dict nel checkpoint di training")
    config = checkpoint.get("config", {})
    missing = validate_checkpoint_config(config, spec)
    model = build_model(spec)
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    model.eval()
    metadata = dict(
        selected_epoch=checkpoint.get("epoch"), best_validation_miou=checkpoint.get("best_miou"),
        configured_training_epochs=config.get("epochs"), checkpoint_config=config,
        missing_metadata_fields=missing, checkpoint_sha256=file_digest(path),
    )
    del checkpoint
    gc.collect()
    return model, metadata


## 9. Inferenza: un modello alla volta
Il manifest di completamento viene scritto solo dopo le 20 predizioni. Una cella interrotta puo' essere rieseguita: i modelli gia' completati e verificati vengono saltati.


In [ ]:
def prediction_paths(spec, sample):
    folder = OUTPUT_DIR / "predictions" / spec["key"]
    return dict(ids=folder / f"{sample['sample_id']}_ids.png",
                color=folder / f"{sample['sample_id']}_color.png")

def model_signature(spec, remote):
    return canonical_digest(dict(
        selection_sha256=SELECTION_SHA256, model_spec=spec,
        checkpoint_id=remote["id"], checkpoint_md5=remote["md5Checksum"],
        checkpoint_size=int(remote["size"]), precision="fp32", batch_size=BATCH_SIZE,
        device_type=device.type, runtime_versions=RUNTIME_VERSIONS,
        inference_code_sha256=INFERENCE_CODE_SHA256,
    ))

def completed_model(spec, signature):
    path = OUTPUT_DIR / "predictions" / spec["key"] / "manifest.json"
    if not path.exists():
        return None
    try:
        manifest = json.loads(path.read_text(encoding="utf-8"))
        if manifest["signature"] != signature or len(manifest["predictions"]) != len(samples):
            return None
        for sample, saved in zip(samples, manifest["predictions"]):
            if saved["sample_id"] != sample["sample_id"]:
                return None
            for name, path in prediction_paths(spec, sample).items():
                if not path.is_file() or file_digest(path) != saved["sha256"][name]:
                    return None
        return manifest
    except (OSError, ValueError, KeyError, TypeError):
        return None

def effective_inference_code_hash():
    # Ignora nome file/numero di esecuzione della cella: il valore resta stabile
    # tra sessioni, ma cambia se si modificano preprocessing, modelli o inferenza.
    def constant(value):
        if hasattr(value, "co_code"):
            return code_payload(value)
        if isinstance(value, (tuple, list)):
            return [constant(item) for item in value]
        if isinstance(value, (set, frozenset)):
            return sorted((constant(item) for item in value), key=lambda item: json.dumps(item, sort_keys=True))
        if isinstance(value, bytes):
            return {"bytes": value.hex()}
        if value is None or isinstance(value, (bool, int, float, str)):
            return value
        return repr(value)

    def code_payload(code):
        return dict(bytecode=code.co_code.hex(), constants=[constant(v) for v in code.co_consts],
                    names=list(code.co_names), variables=list(code.co_varnames),
                    arguments=[code.co_argcount, code.co_posonlyargcount, code.co_kwonlyargcount],
                    flags=code.co_flags, free_variables=list(code.co_freevars), cells=list(code.co_cellvars))

    names = ("SegmentationTransform", "prepare_batch", "colorize_mask", "prepare_selection",
             "ConvBNSelu", "DownsamplingBlock", "ThreeBranchContextAggregation",
             "PartialChannelTransformation", "DualAttentionGuidedDecoder", "LCNet",
             "SMPModelForMars", "SegFormerB0ForMars", "build_model",
             "validate_checkpoint_config", "load_trained_model", "run_one_model")
    payload = {"variant_dilations": VARIANT_DILATIONS}
    for name in names:
        obj = globals()[name]
        if isinstance(obj, type):
            methods = {}
            for method_name, method in obj.__dict__.items():
                method = getattr(method, "__func__", method)
                if hasattr(method, "__code__"):
                    methods[method_name] = code_payload(method.__code__)
            payload[name] = methods
        else:
            payload[name] = code_payload(obj.__code__)
    return canonical_digest(payload)

def run_one_model(service, spec, remote):
    signature = model_signature(spec, remote)
    previous = completed_model(spec, signature) if RESUME_COMPLETED_MODELS else None
    if previous is not None:
        print("Predizioni gia' complete e verificate:", spec["key"])
        return previous
    path = download_checkpoint(service, spec, remote)
    model = images = masks = logits = predictions = None
    try:
        model, checkpoint_metadata = load_trained_model(path, spec)
        model = model.to(device).eval()
        saved_predictions = []
        with torch.inference_mode():
            for start in tqdm(range(0, len(samples), BATCH_SIZE), desc=spec["key"]):
                batch_samples = samples[start:start + BATCH_SIZE]
                rgb_arrays, mask_arrays = [], []
                for sample in batch_samples:
                    with Image.open(OUTPUT_DIR / sample["paths"]["rgb"]) as source:
                        rgb_arrays.append(np.asarray(source.convert("RGB"), dtype=np.uint8).copy())
                    with Image.open(OUTPUT_DIR / sample["paths"]["ground_truth_ids"]) as source:
                        mask_arrays.append(np.asarray(source, dtype=np.uint8).copy())
                images, masks = prepare_batch(
                    dict(image=np.stack(rgb_arrays), mask=np.stack(mask_arrays)), device)
                logits = model(images)
                expected_shape = (len(batch_samples), NUM_CLASSES, *IMAGE_SIZE)
                if tuple(logits.shape) != expected_shape or logits.dtype != torch.float32:
                    raise RuntimeError(f"Output non FP32 o shape errata: {logits.shape}, {logits.dtype}")
                if not torch.isfinite(logits).all():
                    raise RuntimeError(f"{spec['key']}: logits con NaN o Inf")
                predictions = logits.argmax(dim=1).cpu().numpy().astype(np.uint8)
                for sample, labels in zip(batch_samples, predictions):
                    paths = prediction_paths(spec, sample)
                    save_png(paths["ids"], labels)
                    save_png(paths["color"], colorize_mask(labels))
                    saved_predictions.append(dict(
                        sample_id=sample["sample_id"], test_index=sample["test_index"],
                        paths={name: str(p.relative_to(OUTPUT_DIR).as_posix()) for name, p in paths.items()},
                        sha256={name: file_digest(p) for name, p in paths.items()},
                    ))
                images = masks = logits = predictions = None
        manifest = dict(
            schema_version=1, signature=signature, model_spec=spec, remote_checkpoint=remote,
            checkpoint_metadata=checkpoint_metadata, precision="fp32", input_size=list(IMAGE_SIZE),
            selection_sha256=SELECTION_SHA256, runtime_versions=RUNTIME_VERSIONS,
            inference_code_sha256=INFERENCE_CODE_SHA256, predictions=saved_predictions,
        )
        save_json(OUTPUT_DIR / "predictions" / spec["key"] / "manifest.json", manifest)
        return manifest
    finally:
        images = masks = logits = predictions = None
        if model is not None:
            model.cpu()
        del model
        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

INFERENCE_CODE_SHA256 = effective_inference_code_hash()
model_manifests = {}
for number, spec in enumerate(MODELS, start=1):
    print(f"\nModello {number}/{len(MODELS)}: {spec['key']}")
    model_manifests[spec["key"]] = run_one_model(drive_service, spec, REMOTE_CHECKPOINTS[spec["key"]])
save_json(OUTPUT_DIR / "run_manifest.json", dict(
    selection_sha256=SELECTION_SHA256, model_order=[s["key"] for s in MODELS],
    runtime_versions=RUNTIME_VERSIONS,
    model_signatures={key: value["signature"] for key, value in model_manifests.items()},
))
print(f"Completate {len(samples) * len(MODELS)} predizioni; nessun modello rimane sulla GPU.")


## 10. Salvare le 20 griglie 3 x 3
I pannelli mantengono la stessa palette e risoluzione. Le maschere sono mostrate senza interpolazione. PNG a 180 dpi e PDF della figura vengono salvati insieme ai singoli pannelli.


In [ ]:
def render_comparison(sample):
    panels = [
        ("RGB image", OUTPUT_DIR / sample["paths"]["rgb"]),
        ("Ground truth", OUTPUT_DIR / sample["paths"]["ground_truth_color"]),
    ] + [(spec["title"], prediction_paths(spec, sample)["color"]) for spec in MODELS]
    fig, axes = plt.subplots(3, 3, figsize=(10.5, 11.2))
    for axis, (title, path) in zip(axes.flat, panels):
        with Image.open(path) as panel:
            axis.imshow(np.asarray(panel).copy(), interpolation="nearest")
        axis.set_title(title, fontsize=11, pad=5)
        axis.set_axis_off()
    handles = [Patch(facecolor=np.asarray(PALETTE[k]) / 255.0,
                     edgecolor="black", linewidth=0.4, label=CLASS_NAMES[k]) for k in CLASS_NAMES]
    fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=9, frameon=False,
               bbox_to_anchor=(0.5, 0.012))
    fig.subplots_adjust(left=0.015, right=0.985, top=0.975, bottom=0.095, wspace=0.055, hspace=0.13)
    folder = OUTPUT_DIR / "comparisons"
    folder.mkdir(parents=True, exist_ok=True)
    paths = {extension: folder / f"{sample['sample_id']}.{extension}" for extension in ("png", "pdf")}
    try:
        for extension, path in paths.items():
            fig.savefig(path, format=extension, dpi=FIGURE_DPI, facecolor="white")
    finally:
        plt.close(fig)
    return dict(order=sample["order"], test_index=sample["test_index"], sample_id=sample["sample_id"],
                model_signatures={spec["key"]: model_signature(spec, REMOTE_CHECKPOINTS[spec["key"]]) for spec in MODELS},
                paths={extension: str(path.relative_to(OUTPUT_DIR).as_posix()) for extension, path in paths.items()})

# Controlla tutte le mask una sola volta prima di creare le 20 figure.
for spec in MODELS:
    if completed_model(spec, model_signature(spec, REMOTE_CHECKPOINTS[spec["key"]])) is None:
        raise RuntimeError(f"Predizioni mancanti o non valide per {spec['key']}; riesegui inferenza.")
comparisons = [render_comparison(sample) for sample in tqdm(samples, desc="Griglie comparative")]
save_json(OUTPUT_DIR / "comparisons.json", comparisons)
print("Griglie salvate:", OUTPUT_DIR / "comparisons")


## 11. Sfogliare i confronti e scegliere il preferito
Con `sequential` vengono mostrate le 20 griglie, una dopo l'altra come output separati. Il navigatore e' sempre disponibile: usa Precedente/Successiva o il menu, poi **Scegli per la tesi**. La scelta salva figura PNG/PDF, RGB, ground truth e tutte le predizioni in `selected/`; non richiede inferenza.


In [ ]:
def show_comparison(number):
    if not 1 <= int(number) <= len(comparisons):
        raise ValueError(f"Scegli un numero fra 1 e {len(comparisons)}")
    item = comparisons[int(number) - 1]
    print(f"Confronto {item['order']}/{len(comparisons)} | test index {item['test_index']} | {item['sample_id']}")
    display(DisplayImage(filename=str(OUTPUT_DIR / item["paths"]["png"]), width=950))

def make_archive():
    archive_path = OUTPUT_DIR.parent / f"{RUN_TAG}_comparisons.zip"
    included = {"samples", "predictions", "comparisons", "selected"}
    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in sorted(OUTPUT_DIR.rglob("*")):
            relative = path.relative_to(OUTPUT_DIR)
            if path.is_file() and not path.name.endswith(".part") and (
                relative.parts[0] in included or (len(relative.parts) == 1 and path.suffix == ".json")
            ):
                archive.write(path, arcname=str(Path(RUN_TAG) / relative))
    return archive_path

def choose_comparison(number):
    number = int(number)
    if not 1 <= number <= len(samples):
        raise ValueError(f"Scegli un numero fra 1 e {len(samples)}")
    sample, comparison = samples[number - 1], comparisons[number - 1]
    signatures = {spec["key"]: model_signature(spec, REMOTE_CHECKPOINTS[spec["key"]]) for spec in MODELS}
    if comparison["model_signatures"] != signatures:
        raise RuntimeError("La griglia usa predizioni precedenti: riesegui la cella delle griglie.")
    for spec in MODELS:
        if completed_model(spec, model_signature(spec, REMOTE_CHECKPOINTS[spec["key"]])) is None:
            raise RuntimeError(f"Predizioni non valide per {spec['key']}; riesegui inferenza e griglie.")
    folder = OUTPUT_DIR / "selected"
    folder.mkdir(parents=True, exist_ok=True)
    for extension in ("png", "pdf"):
        shutil.copy2(OUTPUT_DIR / comparison["paths"][extension], folder / f"figure_5_1.{extension}")
    for name, source in sample["paths"].items():
        shutil.copy2(OUTPUT_DIR / source, folder / f"{name}.png")
    for spec in MODELS:
        for name, source in prediction_paths(spec, sample).items():
            shutil.copy2(source, folder / f"{spec['key']}_{name}.png")
    choice = dict(order=number, test_index=sample["test_index"], sample_id=sample["sample_id"],
                  selection_sha256=SELECTION_SHA256,
                  model_signatures=signatures)
    save_json(folder / "choice.json", choice)
    (folder / "latex_snippet.tex").write_text(
        "% Copia figure_5_1.pdf nella cartella Images della tesi.\n"
        "\\includegraphics[width=\\textwidth]{Images/figure_5_1.pdf}\n", encoding="utf-8")
    print(f"Scelto confronto {number}, test index {sample['test_index']}. File in {folder}")
    display(FileLink(str(folder / "figure_5_1.png")))
    display(FileLink(str(folder / "figure_5_1.pdf")))
    if CREATE_ZIP:
        display(FileLink(str(make_archive())))
    return choice

if DISPLAY_MODE == "sequential":
    for number in range(1, len(comparisons) + 1):
        show_comparison(number)

try:
    import ipywidgets as widgets
except ImportError:
    if DISPLAY_MODE == "interactive":
        show_comparison(1)
    print("Navigatore non disponibile: usa show_comparison(1) e choose_comparison(1).")
else:
    chooser = widgets.Dropdown(
        options=[(f"{c['order']:02d} - test {c['test_index']}", c["order"]) for c in comparisons],
        value=1, description="Confronto:", layout=widgets.Layout(width="330px"))
    previous_button = widgets.Button(description="Precedente")
    next_button = widgets.Button(description="Successiva")
    select_button = widgets.Button(description="Scegli per la tesi", button_style="success")
    preview = widgets.Output()
    choice_output = widgets.Output()

    def refresh_preview(change=None):
        previous_button.disabled = chooser.value == 1
        next_button.disabled = chooser.value == len(comparisons)
        with preview:
            clear_output(wait=True)
            show_comparison(chooser.value)

    def save_choice(button):
        with choice_output:
            clear_output(wait=True)
            choose_comparison(chooser.value)

    previous_button.on_click(lambda button: setattr(chooser, "value", max(1, chooser.value - 1)))
    next_button.on_click(lambda button: setattr(chooser, "value", min(len(comparisons), chooser.value + 1)))
    select_button.on_click(save_choice)
    chooser.observe(refresh_preview, names="value")
    display(widgets.VBox([widgets.HBox([previous_button, chooser, next_button, select_button]),
                          preview, choice_output]))
    refresh_preview()


## 12. Scelta manuale opzionale e ZIP
Alternativa al pulsante: imposta `CHOSEN_COMPARISON` con il numero 1..20 e riesegui solo questa cella. Lo ZIP contiene tutti i confronti e, se gia' scelta, la figura preferita. I checkpoint non sono inclusi.


In [ ]:
CHOSEN_COMPARISON = None  # Esempio: 7; None lascia la scelta al navigatore.
if CHOSEN_COMPARISON is not None:
    chosen = choose_comparison(CHOSEN_COMPARISON)
elif CREATE_ZIP:
    archive_path = make_archive()
    print("Scarica questo archivio prima di terminare la sessione Kaggle:")
    display(FileLink(str(archive_path)))
print("Output:", OUTPUT_DIR)
